# LankaInsight AI: Prompt Engineering Experiments (Groq API)
This notebook tests the 4 core prompt engineering patterns using the ultra-fast Groq API (Llama 3) before deploying to Streamlit.

**Goals:**
1. Zero-Shot with Delimiters
2. Few-Shot Classification
3. Chain-of-Thought with Inner Monologue
4. Structured JSON Output

Setup & API Initialization

In [16]:
import os
import json
import re
from groq import Groq
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get API key
api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise ValueError("GROQ_API_KEY not found. Please check your .env file.")

# Initialize the Groq client
client = Groq(api_key=api_key)

print("Groq API Client configured successfully!")

Groq API Client configured successfully!


## Pattern 1: Zero-Shot Prompting with Delimiters
**Concept:** Give the model a clear task without examples, but use delimiters (like `"""`) to separate the instruction from the input data.
**Parameters:** 
- `temperature=0.2`: Low randomness for factual, consistent summaries.
- `max_tokens=150`: Keeps the output concise.
- `top_p=0.8`: Nucleus sampling; considers only the most likely tokens, allowing slight natural variety.

In [23]:
news_text = """
The Central Bank of Sri Lanka reported that inflation eased to 4.5% in August, down from 5.1% in July, 
driven by lower food and energy prices. However, the bank warned that global supply chain disruptions 
could cause minor fluctuations in the coming quarter. The rupee has remained relatively stable against 
the US dollar.
"""

zero_shot_prompt = f"""
You are an expert economic analyst. 
Summarize the text provided in the delimiters below into exactly 3 bullet points.
Target audience: High school students in Sri Lanka. Keep the language simple and easy to understand.

Text to summarize:
\"\"\"
{news_text}
\"\"\"
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-20b", # Powerful, fast open-source model
    messages=[{"role": "user", "content": zero_shot_prompt}],
    temperature=0.2,
    max_tokens=150,
    top_p=0.8
)

print("ZERO-SHOT SUMMARY:\n")
print(response.choices[0].message.content)

ZERO-SHOT SUMMARY:

- In August, Sri Lanka’s inflation fell to 4.5 % from 5.1 % in July, mainly because food and energy costs went down.  
- The Central Bank cautions that problems in global supply chains might still cause small price changes next quarter.  
- The Sri Lankan rupee has stayed fairly steady against the U.S. dollar.


## Pattern 2: Few-Shot Prompting
**Concept:** Provide 2-3 examples of `Input -> Output` to teach the model the exact pattern, tone, and format you want, without fine-tuning.
**Parameters:** `temperature=0.1` (Very low to ensure the model strictly mimics the provided examples without adding creative flair).

In [29]:
few_shot_prompt = """
You are a customer service analyzer for a restaurant in Colombo. 
Classify the following customer reviews into exactly ONE of these categories: 
[Food Quality, Service, Ambiance, Pricing].

IMPORTANT: Output ONLY the category name. Do not explain your reasoning.

Examples:
Review: "The kottu was cold and the chicken was undercooked."
Category: Food Quality

Review: "The waiter took 45 minutes to bring the bill, very frustrating."
Category: Service

Review: "A bit expensive for the portion size, but the view of the ocean was nice."
Category: Pricing

Now, classify this new review:
Review: "The acoustic music was too loud, we couldn't even hear each other speak."
Category: 
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": few_shot_prompt}],
    temperature=0.1,
    max_tokens=100  
)

print("FEW-SHOT CLASSIFICATION:\n")
print(response.choices[0].message.content.strip())

FEW-SHOT CLASSIFICATION:

Ambiance


## Pattern 3: Chain-of-Thought (CoT) & Inner Monologue
**Concept:** Force the model to "think step-by-step" to reduce hallucinations. We ask it to output reasoning in `<thinking>` tags, and the final answer in `<final_answer>` tags. Our Python code will hide the `<thinking>` part from the end user.
**Parameters:** `temperature=0.5` (Medium temperature allows the model to explore different logical paths while staying grounded).

In [47]:
cot_prompt = """
You are a Sri Lankan business consultant. 
Analyze the potential impact of a hypothetical 20% fuel price hike on the public transport sector.

Instructions:
1. First, think step-by-step about the direct and indirect effects inside <thinking> tags.
2. Then, provide a clear, professional, 2-sentence summary for a government minister inside <final_answer> tags.
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": cot_prompt}],
    temperature=0.5,
    max_tokens=2000  
)

full_output = response.choices[0].message.content
print("FULL MODEL OUTPUT (Raw):\n")
print(full_output)
print("\n" + "="*50 + "\n")

# --- ROBUST INNER MONOLOGUE PARSING ---
# This regex grabs everything after <final_answer>, even if the model forgets the closing </final_answer> tag
final_answer_match = re.search(r'<final_answer>\s*(.*?)(?:</final_answer>|$)', full_output, re.IGNORECASE | re.DOTALL)

if final_answer_match:
    clean_answer = final_answer_match.group(1).strip()
    print("PRESENTED TO USER (Clean):\n")
    print(clean_answer)
else:
    print("Could not parse final answer. Model did not follow format.")
    print("Raw output was:\n", full_output)

FULL MODEL OUTPUT (Raw):

<thinking>
1. **Direct cost impact**: A 20 % rise in fuel prices translates into a proportional increase in operating expenses for diesel‑powered buses, taxis, and freight vehicles.  
2. **Fare adjustments**: Operators will likely raise fares to preserve margins, which can strain low‑income commuters who rely heavily on public transport.  
3. **Ridership elasticity**: Higher fares typically reduce demand; studies in similar economies show a 5–10 % drop in ridership for every 10 % fare hike, potentially leading to revenue shortfalls.  
4. **Modal shift**: Commuters may switch to cheaper alternatives—walking, cycling, or informal transport—exacerbating traffic congestion and reducing public transport’s market share.  
5. **Inflationary pressure**: Fuel‑related cost increases feed into broader inflation, eroding real incomes and further dampening demand for discretionary travel.  
6. **Employment effects**: Reduced ridership and higher operating costs can trigger

## Pattern 4: Structured Output Parsing (JSON)
**Concept:** Force the LLM to output strictly valid JSON so our backend code can reliably parse it. While Groq doesn't have a native "JSON mode" flag like Gemini, Llama 3 is highly capable of outputting perfect JSON when explicitly instructed and given a schema, especially at `temperature=0.0`.
**Parameters:** `temperature=0.0` (Zero temperature for absolute determinism and strict schema adherence).

In [48]:
job_posting_text = """
Wanted: Junior Data Analyst at TechLanka Solutions. 
Location: Hybrid (Colombo 03 / Remote). 
Requirements: Python, SQL, PowerBI. 
Salary: LKR 120,000 - 150,000 per month. 
Contact: careers@techlanka.lk
"""

structured_prompt = f"""
Extract the key information from the following job posting and format it strictly as a JSON object.
Do not include any markdown formatting (like ```json), do not add any conversational text. Just the raw JSON object.

Required JSON Schema:
{{
  "company_name": "string",
  "role": "string",
  "location": "string",
  "salary_range_lkr": "string",
  "required_skills": ["string", "string"]
}}

Job Posting:
\"\"\"
{job_posting_text}
\"\"\"
"""

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": structured_prompt}],
    temperature=0.0, # Strict determinism
    max_tokens=2000
)

raw_output = response.choices[0].message.content

# Clean up potential markdown formatting just in case the model adds it
clean_json_string = raw_output.strip().removeprefix("```json").removesuffix("```").strip()

print("STRUCTURED JSON OUTPUT:\n")
try:
    parsed_data = json.loads(clean_json_string)
    print(json.dumps(parsed_data, indent=2))
    print(f"\nSuccessfully extracted company: {parsed_data['company_name']}")
except json.JSONDecodeError:
    print("Failed to parse JSON. Raw output was:\n", raw_output)

STRUCTURED JSON OUTPUT:

{
  "company_name": "TechLanka Solutions",
  "role": "Junior Data Analyst",
  "location": "Hybrid (Colombo 03 / Remote)",
  "salary_range_lkr": "LKR 120,000 - 150,000 per month",
  "required_skills": [
    "Python",
    "SQL",
    "PowerBI"
  ]
}

Successfully extracted company: TechLanka Solutions
